In [2]:
# ============================================================================
# SCRIPT: PSD Stacking by Emission Line Profile (Double-Peaked vs Non-DP)
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib
import warnings
from astropy.timeseries import LombScargle

# ---------------------- USER CONFIG (Edit here) ----------------------

base_dir = r"C:\Users\vitma\Documents\Estudos\UFRGS! DO IT!\Projeto Thaisa Rogério\QSO\data\ztffp_final_data_2025" 
csv_meta_path = r"C:\Users\vitma\Documents\Estudos\UFRGS! DO IT!\Projeto Thaisa Rogério\QSO\Cross_15_luiz.csv"

# --- BANDS TO PROCESS ---
BANDS_TO_PROCESS = ['g', 'r']

# Base Output Directories
plots_dir_base = "plots_DP_Profiles"

# --- UNIT AUTOMATION ---
FREQ_UNIT = 'days' # Options: 'days' or 'Hz'

# --- FILTERING VARIABLES ---
MIN_CONTRIBS_PER_FREQ = 1              # Minimum number of contributions per frequency bin

# --- AUTOMATED BIN WIDTH (PHYSICS-BASED) ---
MIN_CADENCE_DAYS = 1.0 
TARGET_BINS = 500 

if FREQ_UNIT == 'Hz':
    time_multiplier = 86400.0
    freq_label = "Hz"
    f_max_nyquist = 1.0 / (2.0 * (MIN_CADENCE_DAYS * 86400.0))
    BIN_WIDTH = f_max_nyquist / TARGET_BINS  
elif FREQ_UNIT == 'days':
    time_multiplier = 1.0
    freq_label = r"day$^{-1}$"
    f_max_nyquist = 1.0 / (2.0 * MIN_CADENCE_DAYS)
    BIN_WIDTH = f_max_nyquist / TARGET_BINS  
else:
    raise ValueError("FREQ_UNIT must be exactly 'days' or 'Hz'")

# Normalization
NORMALIZATION_MODE = 'psd' # 'standard' or 'psd'

# Master Grid filters (Keep to remove outliers)
mass_edges = [7.0, 10.0]
edd_edges  = [-2.5, 0.0]

dpi_save = 200
csv_meta_path = csv_meta_path.replace("\\", "/")

# =============================================================================
# 0. METADATA CROSSMATCH (Done once for all bands)
# =============================================================================
print("Lendo catálogo de metadados...")
meta = pd.read_csv(csv_meta_path)

# Ensure DP_FINAL is treated strictly as a string (fixes pandas boolean auto-parsing)
if 'DP_FINAL' not in meta.columns:
    raise ValueError("A coluna 'DP_FINAL' não foi encontrada no CSV!")
    
meta['DP_FINAL'] = meta['DP_FINAL'].astype(str).str.strip().str.upper()

ra_cat = meta['RA'].values
dec_cat = meta['DEC'].values
logmbh_cat = meta['LOGMBH'].values
logedd_cat = meta['LOGLEDD_RATIO'].values
dp_cat = meta['DP_FINAL'].values

# Group configuration
dp_categories = ['TRUE', 'MAYBE', 'FALSE', 'IDK']
dp_labels = {
    'TRUE': 'DP (Double Peaked)', 
    'MAYBE': 'INT (Intermediate)', 
    'FALSE': 'NDP (Non-Double Peaked)', 
    'IDK': 'IDK (Unknown)'
}
dp_colors = {
    'TRUE': '#d62728',   # Red
    'MAYBE': '#ff7f0e',  # Orange
    'FALSE': '#1f77b4',  # Blue
    'IDK': '#7f7f7f'     # Gray
}

# =============================================================================
# MAIN LOOP OVER BANDS
# =============================================================================
for band in BANDS_TO_PROCESS:
    print(f"\n{'='*60}")
    print(f"PROCESSING BAND: '{band}'")
    print(f"{'='*60}")
    
    band_plots_dir = os.path.join(plots_dir_base, f"band_{band}")
    os.makedirs(band_plots_dir, exist_ok=True)

    # =============================================================================
    # 1. READING AND PRE-CALCULATION FOR CURRENT BAND
    # =============================================================================
    folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and f not in ["plots", "plots_DP_Profiles"]]
    valid_data = []

    for folder in tqdm(folders, desc=f"Filtrando objetos (Band {band})"):
        try:
            ra_f, dec_f = map(float, folder.split('_'))
        except: continue
        
        # Euclidean Crossmatch
        dist = np.sqrt((ra_cat - ra_f)**2 + (dec_cat - dec_f)**2)
        idx = np.argmin(dist)
        if dist[idx] > 1e-4: continue
        
        m, e, dp_val = logmbh_cat[idx], logedd_cat[idx], dp_cat[idx]
        if np.isnan(m) or np.isnan(e): continue

        # Outlier filter
        if m < mass_edges[0] or m >= mass_edges[-1]: continue
        if e < edd_edges[0] or e >= edd_edges[-1]: continue

        # Dynamically find the file for the current band
        txt_files = glob.glob(os.path.join(base_dir, folder, f"*_ztf_{band}_filtered.txt"))
        if not txt_files: continue
        
        try:
            df = pd.read_csv(txt_files[0])
            if len(df) < 10 or 'mjd' not in df.columns or 'flux' not in df.columns: continue
            t = df['mjd'].values
            flux = df['flux'].values
        except: continue
        
        T = t.max() - t.min()
        dt = np.min(np.diff(t))
        
        valid_data.append({
            'label': folder, 't': t, 'flux': flux, 'T': T, 'dt': dt,
            'DP_FINAL': dp_val
        })

    if not valid_data:
        print(f"AVISO: Nenhum dado válido encontrado para a banda '{band}'. Pulando para a próxima...")
        continue

    # Master Grid Limits for this specific band
    T_max = max([d['T'] for d in valid_data])
    dt_min = min([d['dt'] for d in valid_data])

    global_fmin = 1.0 / (T_max * time_multiplier)
    global_fmax = 1.0 / (2.0 * dt_min * time_multiplier)

    N_bins = int(np.ceil((global_fmax - global_fmin) / BIN_WIDTH))
    bin_edges = global_fmin + np.arange(N_bins + 1) * BIN_WIDTH
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    binned_matrix = np.full((len(valid_data), N_bins), np.nan)
    dp_array = np.array([d['DP_FINAL'] for d in valid_data])

    # =============================================================================
    # 2. AUTOPOWER PSD CALCULATION & BINNING
    # =============================================================================
    for i, d in enumerate(tqdm(valid_data, desc=f"Computing Autopower & Binning (Band {band})")):
        t_scaled = (d['t'] - d['t'].min()) * time_multiplier
        flux = d['flux']
        
        y_norm = (flux - np.median(flux)) / np.median(flux)
        
        ls = LombScargle(t_scaled, y_norm)
        freq_auto, power_auto = ls.autopower(normalization=NORMALIZATION_MODE)
        
        inds = np.searchsorted(bin_edges, freq_auto, side='right') - 1
        valid_mask = (inds >= 0) & (inds < N_bins)
        
        if np.any(valid_mask):
            bid = inds[valid_mask]
            pw = power_auto[valid_mask]
            
            sums = np.bincount(bid, weights=pw, minlength=N_bins)
            counts = np.bincount(bid, minlength=N_bins)
            
            means = np.full(N_bins, np.nan)
            mask = counts > 0
            means[mask] = sums[mask] / counts[mask]
            binned_matrix[i, :] = means

    # =============================================================================
    # 3. STACKING BY DP_FINAL (DOUBLE PEAKED MORPHOLOGY)
    # =============================================================================
    plt.figure(figsize=(10, 7))
    plotted_curves = 0

    for dp_val in dp_categories:
        # Find indices of objects matching the current category
        idxs = np.where(dp_array == dp_val)[0]
        nobj = len(idxs)
        
        if nobj < 1: 
            continue
            
        arr = binned_matrix[idxs, :]
        contrib_counts = np.sum(~np.isnan(arr), axis=0)
        good_mask = contrib_counts >= MIN_CONTRIBS_PER_FREQ
        
        if np.sum(good_mask) < 2: 
            continue
            
        # Calculate the median PSD for this group
        med = np.full(N_bins, np.nan)
        med[good_mask] = np.nanmedian(arr[:, good_mask], axis=0)
        
        # Plotting arrays
        x = np.log10(bin_centers[good_mask])
        y = np.log10(med[good_mask])
        
        label_str = f"{dp_labels.get(dp_val, dp_val)} (N={nobj})"
        color_str = dp_colors.get(dp_val, 'black')
        
        # Emphasize the TRUE (DP) line with a slightly thicker curve or higher z-order
        z_order = 10 if dp_val == 'TRUE' else 5
        line_w = 2.5 if dp_val == 'TRUE' else 1.8
        
        plt.plot(x, y, label=label_str, color=color_str, linewidth=line_w, zorder=z_order, alpha=0.9)
        plotted_curves += 1
        
    if plotted_curves > 0:
        plt.xlabel(f"log10(frequency [{freq_label}])", fontweight='bold', fontsize=12)
        plt.ylabel("log10(Power)", fontweight='bold', fontsize=12)
        plt.title(f"Stacked PSD by Emission Line Profile (Band {band})", fontweight='bold', fontsize=14)
        plt.legend(loc='best', fontsize='medium')
        plt.grid(True, ls=':', alpha=0.4)
        plt.tight_layout()
        
        out_plot_name = os.path.join(band_plots_dir, f"stack_DP_Profiles_band_{band}.png")
        plt.savefig(out_plot_name, dpi=dpi_save)
        print(f"Gráfico salvo com sucesso em: {out_plot_name}")
    else:
        print(f"Aviso: Não houve curvas suficientes para plotar o gráfico na banda {band}.")
        
    plt.close()

print("\nProcessamento de perfis de Duplo Pico completo para todas as bandas!")

Lendo catálogo de metadados...

PROCESSING BAND: 'g'


Computing Autopower & Binning (Band g): 100%|██████████| 465/465 [00:03<00:00, 151.23it/s]


Gráfico salvo com sucesso em: plots_DP_Profiles\band_g\stack_DP_Profiles_band_g.png

PROCESSING BAND: 'r'


Computing Autopower & Binning (Band r): 100%|██████████| 462/462 [00:03<00:00, 129.51it/s]


Gráfico salvo com sucesso em: plots_DP_Profiles\band_r\stack_DP_Profiles_band_r.png

Processamento de perfis de Duplo Pico completo para todas as bandas!
